# ONIA · Regresia liniară

Notebook de lucru pentru lecția **2.1. Regresia liniară**. Încearcă să explici fiecare pas înainte de a executa celula.

Concepte: predicții, MSE/MAE, soluția OLS, pseudoinversă, gradient descent, regresie polinomială, Ridge, Lasso, Elastic Net și scikit-learn.

## Predicții pentru câteva valori ale lui x

In [ ]:
w = 2
b = 1

for x in [0, 1, 3, 5]:
    y_estimat = w * x + b
    print(f"x={x}, predicție={y_estimat}")


## Calculul erorii pătratice medii de la zero

In [ ]:
def mse(y_real, y_prezis):
    n = len(y_real)
    suma = 0
    for i in range(n):
        eroare = y_prezis[i] - y_real[i]
        suma += eroare * eroare
    return suma / n

print(mse([2, 4, 6], [3, 2, 7]))  # 2.0


## MAE, MSE și RMSE din liste Python

In [ ]:
from math import sqrt

def metrici(y_real, y_prezis):
    n = len(y_real)
    erori = [y_prezis[i] - y_real[i] for i in range(n)]
    mae = sum(abs(e) for e in erori) / n
    mse = sum(e * e for e in erori) / n
    return mae, mse, sqrt(mse)

print(metrici([3, 4, 5], [4, 2, 5]))
# (1.0, 1.666..., 1.290...)


## Soluția exactă OLS pentru o caracteristică (Python simplu)

In [ ]:
def regresie_ols(x, y):
    n = len(x)
    if n != len(y) or n < 2:
        raise ValueError("Sunt necesare cel puțin două perechi")

    media_x = sum(x) / n
    media_y = sum(y) / n
    sus = sum((x[i] - media_x) * (y[i] - media_y)
              for i in range(n))
    jos = sum((x[i] - media_x) ** 2 for i in range(n))
    if jos == 0:
        raise ValueError("Toate valorile lui x sunt identice")

    w = sus / jos
    b = media_y - w * media_x
    return w, b

print(regresie_ols([1, 2, 3], [2, 3, 5]))
# (1.5, 0.3333333333333335)


## Soluție cu pseudoinversă NumPy

In [ ]:
import numpy as np

# Coloanele 2 și 3 sunt identice.
X = np.array([
    [1, 1, 1],
    [1, 2, 2],
    [1, 3, 3]
], dtype=float)
y = np.array([2, 3, 4], dtype=float)

theta = np.linalg.pinv(X) @ y
print(np.round(theta, 3))  # [1.  0.5 0.5]
print(X @ theta)           # [2. 3. 4.]


## Antrenarea din zero, cu Python de bază

In [ ]:
def antreneaza_linie(x, y, eta=0.05, epoci=2000):
    n = len(x)
    w, b = 0.0, 0.0
    istoric = []

    for epoca in range(epoci):
        gradient_w = 0.0
        gradient_b = 0.0
        eroare_totala = 0.0

        for i in range(n):
            predictie = w * x[i] + b
            eroare = predictie - y[i]
            gradient_w += eroare * x[i]
            gradient_b += eroare
            eroare_totala += eroare ** 2

        gradient_w = 2 * gradient_w / n
        gradient_b = 2 * gradient_b / n
        w -= eta * gradient_w
        b -= eta * gradient_b
        if epoca % 200 == 0:
            istoric.append(eroare_totala / n)

    return w, b, istoric

x = [1, 2, 3]
y = [2, 3, 5]
w, b, erori = antreneaza_linie(x, y, eta=0.05)
print(round(w, 3), round(b, 3))
print([round(e, 3) for e in erori[:4]])


## Regresie cu două sau mai multe caracteristici, de la zero

In [ ]:
def antreneaza(X, y, eta=0.05, epoci=3000):
    n = len(X)
    k = len(X[0])
    w = [0.0] * k
    b = 0.0

    for epoca in range(epoci):
        gw = [0.0] * k
        gb = 0.0
        for i in range(n):
            predictie = b
            for j in range(k):
                predictie += w[j] * X[i][j]
            eroare = predictie - y[i]
            gb += eroare
            for j in range(k):
                gw[j] += eroare * X[i][j]

        b -= eta * (2 * gb / n)
        for j in range(k):
            w[j] -= eta * (2 * gw[j] / n)
    return w, b

X = [[0, 0], [1, 0], [0, 1], [1, 1]]
y = [2, 5, 4, 7]  # y = 3*x1 + 2*x2 + 2
w, b = antreneaza(X, y)
print([round(v, 2) for v in w], round(b, 2))
# [3.0, 2.0] 2.0


## Compararea erorilor de antrenare și validare

In [ ]:
def calculeaza_mse(x, y, w, b):
    total = 0
    for i in range(len(x)):
        eroare = w * x[i] + b - y[i]
        total += eroare * eroare
    return total / len(x)

x_train = [0, 1, 2, 3]
y_train = [1, 3, 5, 7]
x_val = [4, 5]
y_val = [9, 12]

print(calculeaza_mse(x_train, y_train, 2, 1))  # 0.0
print(calculeaza_mse(x_val, y_val, 2, 1))      # 0.5


## Ridge pentru o caracteristică — implementare de la zero

In [ ]:
def ridge_1d(x, y, lam):
    n = len(x)
    mx = sum(x) / n
    my = sum(y) / n
    sxy = sum((x[i] - mx) * (y[i] - my)
              for i in range(n))
    sxx = sum((xi - mx) ** 2 for xi in x)
    if sxx + lam == 0:
        raise ValueError("Nu putem determina o pantă unică")
    w = sxy / (sxx + lam)
    b = my - w * mx
    return w, b

x = [1, 2, 3]
y = [2, 3, 5]
print(ridge_1d(x, y, 0))   # aceeași pantă ca OLS
print(ridge_1d(x, y, 2))   # panta este micșorată


## Lasso unidimensional — soluție exactă, fără biblioteci

In [ ]:
def prag_moale(z, prag):
    if z > prag:
        return z - prag
    if z < -prag:
        return z + prag
    return 0.0

def lasso_1d(x, y, lam):
    n = len(x)
    mx, my = sum(x) / n, sum(y) / n
    sxy = sum((x[i] - mx) * (y[i] - my)
              for i in range(n))
    sxx = sum((xi - mx) ** 2 for xi in x)
    if sxx == 0:
        raise ValueError("Toate valorile lui x sunt identice")
    w = prag_moale(sxy, lam / 2) / sxx
    return w, my - w * mx

print(lasso_1d([1, 2, 3], [2, 3, 5], 10))
# (0.0, 3.3333333333333335)


## Elastic Net cu o singură caracteristică, de la zero

In [ ]:
def prag_moale(z, prag):
    if z > prag:
        return z - prag
    if z < -prag:
        return z + prag
    return 0.0

def elastic_net_1d(x, y, lam1, lam2):
    n = len(x)
    mx = sum(x) / n
    my = sum(y) / n
    sxy = sum((x[i] - mx) * (y[i] - my)
              for i in range(n))
    sxx = sum((xi - mx) ** 2 for xi in x)
    w = prag_moale(sxy, lam1 / 2) / (sxx + lam2)
    b = my - w * mx
    return w, b

print(elastic_net_1d([1, 2, 3], [2, 3, 5], 1, 2))
# (0.625, 2.0833333333333335)


## LinearRegression cu evaluare pe test

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = np.array([[1], [2], [3], [4], [5], [6], [7], [8]], dtype=float)
y = np.array([2.4, 3.1, 4.4, 5.0, 6.2, 7.1, 8.1, 9.4])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Panta:", model.coef_[0])
print("Intercept:", model.intercept_)
print("MAE:", mean_absolute_error(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R²:", r2_score(y_test, pred))
print("Predicția pentru x=9:", model.predict([[9]])[0])


## Comparăm cele patru modele pe aceleași date

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = np.array([[0, 2], [1, 1], [2, 3], [3, 2],
              [4, 5], [5, 3], [6, 6], [7, 4]], dtype=float)
y = np.array([2.1, 3.0, 5.9, 6.8, 10.3, 10.4, 13.9, 14.4])

modele = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.05, max_iter=20000),
    "ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.5,
                             max_iter=20000),
}

for nume, estimator in modele.items():
    # Normalizarea se învață pe setul de antrenare.
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(X, y)
    print(nume, model.predict([[4, 4]])[0])


## Regresie polinomială de gradul 3, cu Ridge

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X = np.linspace(-2, 2, 40).reshape(-1, 1)
y = 1 + 0.5 * X[:, 0] + 2 * X[:, 0] ** 2
y += np.random.default_rng(42).normal(0, 0.3, size=len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    Ridge(alpha=1.0)
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MSE test:", round(mean_squared_error(y_test, pred), 3))
